In [48]:
!pip install -q streamlit pyjwt pyngrok

import os
from google.colab import userdata

os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write("""[theme]
base = "light"
primaryColor = "#be185d"
backgroundColor = "#fdf2f8"
secondaryBackgroundColor = "#ffffff"
textColor = "#1f2937"
""")

for name in ["JWT_SECRET", "NGROK_AUTHENTICATION", "EMAIL_ADDRESS",
             "EMAIL_PASSWORD", "ADMIN_PASSWORD"]:
    try:
        userdata.get(name)
        print(name, "-> OK")
    except Exception as e:
        print(name, "-> MISSING")

JWT_SECRET -> OK
NGROK_AUTHENTICATION -> OK
EMAIL_ADDRESS -> OK
EMAIL_PASSWORD -> OK
ADMIN_PASSWORD -> OK


In [49]:
%%writefile app.py
import streamlit as st
import sqlite3, re, hashlib, os, time, random, smtplib, jwt
from datetime import datetime, timedelta, timezone
from email.mime.text import MIMEText

JWT_SECRET = os.environ["JWT_SECRET"]
EMAIL_ADDRESS = os.environ["EMAIL_ADDRESS"]
EMAIL_PASSWORD = os.environ["EMAIL_PASSWORD"]
ADMIN_USER = "admin"
ADMIN_PASSWORD = os.environ["ADMIN_PASSWORD"]

QUESTIONS = [
    "What was your first pet's name?",
    "What is your mother's maiden name?",
    "What was the name of your first school?",
    "What is your favourite food?",
    "In which city were you born?",
]

# ---------- Database ----------
def db():
    conn = sqlite3.connect("users.db")
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        username TEXT PRIMARY KEY, email TEXT, password TEXT,
        salt TEXT, question TEXT, answer TEXT)""")
    return conn

def hash_text(text, salt):
    return hashlib.sha256((salt + text).encode()).hexdigest()

# ---------- Validation ----------
def valid_email(email):
    return re.fullmatch(r"[A-Za-z]{2,}[A-Za-z0-9._]*@[A-Za-z]{2,}\.[A-Za-z]{2,}", email) is not None

def valid_password(pw):
    if len(pw) < 8: return "Password must be at least 8 characters."
    if not re.search(r"[A-Z]", pw): return "Add at least one uppercase letter."
    if not re.search(r"[a-z]", pw): return "Add at least one lowercase letter."
    if not re.search(r"[0-9]", pw): return "Add at least one number."
    if not re.search(r"[^A-Za-z0-9]", pw): return "Add at least one special symbol."
    return None

# ---------- JWT ----------
def make_token(username):
    payload = {"sub": username, "exp": datetime.now(timezone.utc) + timedelta(minutes=30)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def check_token(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])["sub"]
    except Exception:
        return None

# ---------- OTP email ----------
def send_otp(to_email, otp):
    msg = MIMEText(f"Your OTP is {otp}. It is valid for 5 minutes.")
    msg["Subject"] = "Your OTP - Infosys Auth App"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_email
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.send_message(msg)

Overwriting app.py


In [50]:
%%writefile -a app.py

st.set_page_config(page_title="Infosys Auth Portal", page_icon="🔐")

st.markdown("""
<style>
header[data-testid="stHeader"], #MainMenu, footer { display: none; }
.stApp { background: #fdf2f8; }
.block-container { max-width: 780px; padding-top: 1.2rem; }

.st-key-navbar {
    background: #ffffff;
    border: 1px solid #fbcfe8;
    border-radius: 12px;
    padding: 10px 16px;
    margin-bottom: 28px;
    box-shadow: 0 1px 3px rgba(157, 23, 77, 0.10);
}
.st-key-navbar .brand {
    color: #9d174d;
    font-weight: 800;
    font-size: 1.1rem;
    padding-top: 6px;
}
.st-key-navbar .stButton > button {
    background: transparent;
    color: #9d174d;
    border: none;
    box-shadow: none;
    font-weight: 600;
    white-space: nowrap;
}
.st-key-navbar .stButton > button:hover {
    background: #fce7f3;
    color: #9d174d;
}

h1 {
    color: #1f2937 !important;
    font-weight: 700 !important;
    font-size: 1.9rem !important;
    padding-bottom: 0.4rem;
}

div[data-testid="stForm"] {
    background: #ffffff;
    border: 1px solid #fbcfe8;
    border-radius: 12px;
    padding: 28px;
    box-shadow: 0 4px 14px rgba(157, 23, 77, 0.08);
}

div[data-testid="stFormSubmitButton"] > button,
.stButton > button {
    background: #be185d;
    color: #ffffff;
    border: none;
    border-radius: 8px;
    font-weight: 600;
    padding: 0.5rem 1.4rem;
}
div[data-testid="stFormSubmitButton"] > button p,
.stButton > button p { color: inherit; }
div[data-testid="stFormSubmitButton"] > button:hover,
.stButton > button:hover {
    background: #9d174d;
    color: #ffffff;
}

.stTextInput input {
    background: #ffffff;
    border: 1px solid #e5c9d6;
    border-radius: 8px;
}
</style>
""", unsafe_allow_html=True)

defaults = {"page": "login", "token": None, "admin": False, "otp": None,
            "otp_email": None, "otp_time": 0, "verified_user": None, "msg": None}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

def go(page):
    st.session_state.page = page
    st.rerun()

# ---------- SIGNUP ----------
def signup_page():
    st.title("Create your account")
    with st.form("signup"):
        username = st.text_input("Username")
        email = st.text_input("Email")
        pw = st.text_input("Password", type="password")
        cpw = st.text_input("Confirm Password", type="password")
        question = st.selectbox("Security Question", QUESTIONS)
        answer = st.text_input("Security Answer")
        ok = st.form_submit_button("Sign Up")
    if ok:
        if not all([username.strip(), email.strip(), pw, cpw, answer.strip()]):
            st.error("All fields are mandatory."); return
        if not valid_email(email.strip()):
            st.error("Enter a valid email (like ab@cd.ef)."); return
        err = valid_password(pw)
        if err:
            st.error(err); return
        if pw != cpw:
            st.error("Passwords do not match."); return
        conn = db()
        if conn.execute("SELECT 1 FROM users WHERE username=?", (username.strip(),)).fetchone():
            st.error("Username already exists. Choose another."); return
        if conn.execute("SELECT 1 FROM users WHERE email=?", (email.strip().lower(),)).fetchone():
            st.error("Email already registered."); return
        salt = os.urandom(8).hex()
        conn.execute("INSERT INTO users VALUES (?,?,?,?,?,?)",
                     (username.strip(), email.strip().lower(), hash_text(pw, salt),
                      salt, question, hash_text(answer.strip().lower(), salt)))
        conn.commit()
        st.session_state.msg = "Signup successful. Please log in."
        go("login")

Appending to app.py


In [ ]:
%%writefile -a app.py

# ---------- LOGIN ----------
def login_page():
    st.title("Welcome back")
    if st.session_state.msg:
        st.success(st.session_state.msg)
        st.session_state.msg = None
    with st.form("login"):
        ident = st.text_input("Username or Email")
        pw = st.text_input("Password", type="password")
        ok = st.form_submit_button("Login")
    if ok:
        if not ident.strip() or not pw:
            st.error("Both fields are mandatory.")
        else:
            row = db().execute(
                "SELECT username, password, salt FROM users WHERE username=? OR email=?",
                (ident.strip(), ident.strip().lower())).fetchone()
            if row and hash_text(pw, row[2]) == row[1]:
                st.session_state.token = make_token(row[0])
                go("dashboard")
            else:
                st.error("Invalid credentials.")
    st.caption("New here? Use Sign Up in the menu above.")

# ---------- DASHBOARD ----------
def dashboard_page():
    user = check_token(st.session_state.token) if st.session_state.token else None
    if not user:
        st.session_state.token = None
        st.session_state.msg = "Session invalid or expired. Please log in."
        go("login")
    st.title("Dashboard")
    st.success(f"Welcome, {user}!")
    if st.button("Logout"):
        st.session_state.token = None
        go("login")

In [ ]:
%%writefile -a app.py

# ---------- FORGOT PASSWORD ----------
def reset_form(username):
    with st.form("reset"):
        npw = st.text_input("New Password", type="password")
        cpw = st.text_input("Confirm New Password", type="password")
        ok = st.form_submit_button("Reset Password")
    if ok:
        if not npw or not cpw:
            st.error("Both fields are mandatory."); return
        err = valid_password(npw)
        if err:
            st.error(err); return
        if npw != cpw:
            st.error("Passwords do not match."); return
        conn = db()
        salt = conn.execute("SELECT salt FROM users WHERE username=?", (username,)).fetchone()[0]
        conn.execute("UPDATE users SET password=? WHERE username=?", (hash_text(npw, salt), username))
        conn.commit()
        st.session_state.verified_user = None
        st.session_state.msg = "Password reset successful. Please log in."
        go("login")

def forgot_page():
    st.title("Forgot Password")
    if st.session_state.verified_user:
        st.success("Verified. Set a new password.")
        reset_form(st.session_state.verified_user)
        if st.button("Cancel"):
            st.session_state.verified_user = None
            go("login")
        return
    tab1, tab2 = st.tabs(["Security Question", "OTP via Email"])
    with tab1:
        uname = st.text_input("Username (press Enter)", key="fq_user")
        if uname.strip():
            row = db().execute("SELECT question, answer, salt FROM users WHERE username=?",
                               (uname.strip(),)).fetchone()
            if row:
                st.info(row[0])
                ans = st.text_input("Your answer", key="fq_ans")
                if st.button("Verify Answer"):
                    if not ans.strip():
                        st.error("Answer is mandatory.")
                    elif hash_text(ans.strip().lower(), row[2]) == row[1]:
                        st.session_state.verified_user = uname.strip()
                        st.rerun()
                    else:
                        st.error("Incorrect answer.")
            else:
                st.error("Username not found.")
    with tab2:
        email = st.text_input("Registered Email", key="fo_email")
        if st.button("Send OTP"):
            if not email.strip():
                st.error("Email is mandatory.")
            elif not valid_email(email.strip()):
                st.error("Enter a valid email.")
            else:
                e = email.strip().lower()
                if not db().execute("SELECT 1 FROM users WHERE email=?", (e,)).fetchone():
                    st.error("Email not registered.")
                else:
                    otp = str(random.SystemRandom().randint(100000, 999999))
                    try:
                        send_otp(e, otp)
                        st.session_state.otp = otp
                        st.session_state.otp_email = e
                        st.session_state.otp_time = time.time()
                        st.success("OTP sent. Check your email.")
                    except Exception:
                        st.error("Could not send OTP. Check the email secrets.")
        if st.session_state.otp:
            code_in = st.text_input("Enter OTP", key="fo_code")
            if st.button("Verify OTP"):
                if not code_in.strip():
                    st.error("OTP is mandatory.")
                elif time.time() - st.session_state.otp_time > 300:
                    st.session_state.otp = None
                    st.error("OTP expired. Send a new one.")
                elif code_in.strip() == st.session_state.otp:
                    u = db().execute("SELECT username FROM users WHERE email=?",
                                     (st.session_state.otp_email,)).fetchone()[0]
                    st.session_state.verified_user = u
                    st.session_state.otp = None
                    st.rerun()
                else:
                    st.error("Incorrect OTP.")

In [ ]:
%%writefile -a app.py

# ---------- ADMIN ----------
def admin_page():
    if st.session_state.admin:
        st.title("Admin Dashboard")
        rows = db().execute("SELECT username, email FROM users").fetchall()
        st.write(f"Total users: {len(rows)}")
        st.table([{"Username": r[0], "Email": r[1]} for r in rows])
        if st.button("Admin Logout"):
            st.session_state.admin = False
            go("login")
        return
    st.title("Admin Login")
    with st.form("admin_form"):
        u = st.text_input("Admin Username")
        p = st.text_input("Admin Password", type="password")
        ok = st.form_submit_button("Login")
    if ok:
        if not u.strip() or not p:
            st.error("Both fields are mandatory.")
        elif u == ADMIN_USER and p == ADMIN_PASSWORD:
            st.session_state.admin = True
            st.rerun()
        else:
            st.error("Invalid admin credentials.")

# ---------- TOP MENU ----------
def navbar():
    logged_in = (st.session_state.page == "dashboard" or
                 (st.session_state.page == "admin" and st.session_state.admin))
    with st.container(key="navbar"):
        cols = st.columns([2.4, 1, 1, 1.6, 1.4])
        cols[0].markdown('<div class="brand">Infosys Auth Portal</div>', unsafe_allow_html=True)
        if not logged_in:
            if cols[1].button("Login", key="nav_login", use_container_width=True):
                go("login")
            if cols[2].button("Sign Up", key="nav_signup", use_container_width=True):
                go("signup")
            if cols[3].button("Forgot Password", key="nav_forgot", use_container_width=True):
                go("forgot")
            if cols[4].button("Admin", key="nav_admin", use_container_width=True):
                go("admin")

# ---------- ROUTER ----------
pages = {"login": login_page, "signup": signup_page, "dashboard": dashboard_page,
         "forgot": forgot_page, "admin": admin_page}
navbar()
pages[st.session_state.page]()

In [ ]:
import os, subprocess, time
from google.colab import userdata
from pyngrok import ngrok

for n in ["JWT_SECRET", "NGROK_AUTHENTICATION", "EMAIL_ADDRESS",
          "EMAIL_PASSWORD", "ADMIN_PASSWORD"]:
    os.environ[n] = userdata.get(n)

os.system("pkill -f streamlit")
ngrok.kill()
time.sleep(2)
ngrok.set_auth_token(os.environ["NGROK_AUTHENTICATION"])
subprocess.Popen(["streamlit", "run", "app.py",
                  "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)
print("Open this link:", ngrok.connect(8501))